# Feature engineering for water quality anomaly detection

This notebook transforms raw watershed monitoring data into the feature matrix
used by the anomaly detection ensemble. Steps include:

1. Loading and cleaning multi-parameter readings
2. Pivoting from long to wide form
3. StandardScaler normalization
4. Z-score computation per parameter and site
5. Temporal feature extraction
6. Parameter distribution inspection

In [ ]:
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, '..')
from src.data_loader import (
    fetch_water_quality_data,
    preprocess,
    pivot_parameters,
    add_rolling_statistics,
    add_rate_of_change,
    add_zscore_features,
    KEY_PARAMETERS,
)

pd.set_option('display.max_columns', 50)
print('Setup complete.')

## 1. Load and clean watershed data

The raw data comes from Calgary's watershed monitoring program. Each row is a
single parameter measurement at a sample site on a given date.

In [ ]:
raw = fetch_water_quality_data(limit=50000)
print(f'Raw records: {len(raw):,}')
print(f'Columns: {list(raw.columns)}')
raw.head()

In [ ]:
df = preprocess(raw)
print(f'After cleaning: {len(df):,} records')
print(f'Sites: {df["sample_site"].nunique()}')
print(f'Parameters: {df["parameter"].nunique()}')
print(f'Date range: {df["sample_date"].min()} to {df["sample_date"].max()}')

## 2. Handle multi-parameter readings

The long-form data has one row per (site, date, parameter). We pivot to wide
form so each parameter becomes its own column, giving one row per (site, date).

In [ ]:
# Parameter frequency
param_counts = df['parameter'].value_counts().head(20).reset_index()
param_counts.columns = ['Parameter', 'Count']

fig = px.bar(param_counts, x='Count', y='Parameter', orientation='h',
             title='Top 20 parameters by measurement count',
             color_discrete_sequence=['teal'])
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=500)
fig.show()

In [ ]:
wide = pivot_parameters(df)
print(f'Wide-form shape: {wide.shape}')
print(f'Parameter columns: {[c for c in wide.columns if c not in ["sample_site", "sample_date"]][:15]}')
wide.head()

## 3. StandardScaler normalization

Isolation Forest and LOF are distance-based and sensitive to feature scale.
We apply `StandardScaler` to the numeric parameter columns so they all have
zero mean and unit variance.

In [ ]:
exclude = {'sample_site', 'sample_date', 'year', 'month'}
numeric_cols = [c for c in wide.columns if c not in exclude and wide[c].dtype in ('float64', 'int64')]

scaler = StandardScaler()
scaled_values = scaler.fit_transform(wide[numeric_cols].fillna(0))
scaled_df = pd.DataFrame(scaled_values, columns=numeric_cols, index=wide.index)

print(f'Scaled {len(numeric_cols)} parameter columns')
print('Mean per column (should be ~0):', scaled_df.mean().round(6).head())
print('Std per column (should be ~1):', scaled_df.std().round(6).head())

## 4. Z-score computation

Z-scores are computed per parameter within each site using
`add_zscore_features`. Values beyond +/- 3 standard deviations are strong
candidates for anomalies.

In [ ]:
wide_z = add_zscore_features(wide)

zscore_cols = [c for c in wide_z.columns if c.endswith('_zscore')]
print(f'Z-score columns created: {len(zscore_cols)}')
wide_z[zscore_cols].describe().T.head(10)

In [ ]:
# Count readings with |z| > 3 per parameter
extreme_counts = {}
for col in zscore_cols:
    count = (wide_z[col].abs() > 3).sum()
    extreme_counts[col.replace('_zscore', '')] = count

extreme_df = pd.DataFrame(list(extreme_counts.items()), columns=['Parameter', 'Count |z| > 3'])
extreme_df = extreme_df.sort_values('Count |z| > 3', ascending=False).head(15)

fig = px.bar(extreme_df, x='Count |z| > 3', y='Parameter', orientation='h',
             title='Readings with |z-score| > 3 by parameter',
             color_discrete_sequence=['crimson'])
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=450)
fig.show()

## 5. Temporal feature extraction

We add rolling statistics (7-day and 30-day windows) and rate-of-change
features to capture temporal dynamics in water quality.

In [ ]:
wide_feat = add_rolling_statistics(wide, windows=(7, 30))
wide_feat = add_rate_of_change(wide_feat)

rolling_cols = [c for c in wide_feat.columns if '_roll_' in c]
roc_cols = [c for c in wide_feat.columns if c.endswith('_roc')]

print(f'Rolling statistic columns: {len(rolling_cols)}')
print(f'Rate-of-change columns:    {len(roc_cols)}')
print(f'Total features: {wide_feat.shape[1]}')

In [ ]:
# Show example rolling features for one site
example_site = wide_feat['sample_site'].value_counts().index[0]
site_data = wide_feat[wide_feat['sample_site'] == example_site].copy()

# Pick a parameter that has rolling columns
sample_param = numeric_cols[0] if numeric_cols else None
if sample_param and f'{sample_param}_roll_mean_30d' in site_data.columns:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=site_data['sample_date'], y=site_data[sample_param],
                             mode='markers', name='Raw', marker=dict(size=4)))
    fig.add_trace(go.Scatter(x=site_data['sample_date'], y=site_data[f'{sample_param}_roll_mean_30d'],
                             mode='lines', name='30-day rolling mean'))
    fig.update_layout(title=f'{sample_param} at site {example_site}: raw vs. rolling mean',
                      xaxis_title='Date', yaxis_title=sample_param, height=400)
    fig.show()

## 6. Parameter distributions

Histograms of the key water quality parameters to check for skewness and
identify natural value ranges.

In [ ]:
available_key = [p for p in KEY_PARAMETERS if p in wide.columns]
if not available_key:
    available_key = numeric_cols[:5]

n = len(available_key)
cols_per_row = min(n, 3)
rows = (n + cols_per_row - 1) // cols_per_row

fig = make_subplots(rows=rows, cols=cols_per_row, subplot_titles=available_key)
for i, param in enumerate(available_key):
    r, c = i // cols_per_row + 1, i % cols_per_row + 1
    fig.add_trace(go.Histogram(x=wide[param].dropna(), name=param,
                               marker_color='teal', showlegend=False),
                  row=r, col=c)
fig.update_layout(title='Key parameter distributions', height=300 * rows)
fig.show()

In [ ]:
# Correlation between key parameters
if len(available_key) >= 2:
    corr = wide[available_key].corr()
    fig = px.imshow(corr, text_auto='.2f', title='Correlation between key parameters',
                    color_continuous_scale='RdBu_r', zmin=-1, zmax=1)
    fig.update_layout(height=450, width=500)
    fig.show()

## 7. Final feature matrix summary

In [ ]:
# Combine all features into the final wide dataframe
final = add_zscore_features(wide_feat)

all_feature_cols = [c for c in final.columns if c not in {'sample_site', 'sample_date', 'year', 'month'}]
print(f'Final feature matrix: {final.shape[0]:,} rows x {len(all_feature_cols)} features')
print(f'Missing values per feature (top 10):')
final[all_feature_cols].isnull().sum().sort_values(ascending=False).head(10)

## Summary

Feature engineering steps completed:

1. Cleaned raw readings and parsed dates, yielding one record per (site, date, parameter).
2. Pivoted to wide form so each parameter is a separate column.
3. Applied `StandardScaler` normalization for distance-based detectors.
4. Computed per-site z-scores for statistical thresholding.
5. Added 7-day and 30-day rolling means/stds and first-difference rate-of-change features.
6. Inspected parameter distributions and correlations.

The resulting feature matrix feeds into the Isolation Forest, LOF, and ensemble
models in `03_modeling.ipynb`.